# 03_01 — TabEBM per-member (01 corner 와 bit-exact 일치)

**목적**: 01 에서 만든 K 개 ensemble member 와 **완전히 같은 4-corner 세트** 위에서  
TabEBM 표준 SGLD 를 K 번 돌려 K 개 sample set 생성.

- 01 의 corner_seed = `split_seed + k` (k = 0..K-1) 와 같은 숫자를 TabEBM.generate 의 seed 에 전달
- `ensemble_methods.build_surrogate_data` 가 MT19937 (RandomState) 로 바뀌었기 때문에 corner **set 단위 일치**
- 결과 파일: `samples/split_{i}/tabebm_m{k}/c{c}.npy`

04 에서 비교:
- `tabebm_m0` = 01 멤버 0 corner 위 단일 TabEBM
- `mean(tabebm_m0..m{K-1})` = K 멤버 평균 (pandas `.mean(axis=1)`)
- `vp_*` = 같은 K 멤버의 VP-SGLD aggregation

→ **같은 corner 풀 위에서 aggregation 방식 (단일 vs K-mean vs VP μ̂Σ̂) 차이** 공정 측정.

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as _mp
try: _mp.set_start_method('spawn', force=True)
except RuntimeError: pass

os.chdir('/home/work/JooKyung/TabEBM')
sys.path.insert(0, 'experiments'); sys.path.insert(0, 'src')

import numpy as np

def _fmt(s):
    s = int(s); m, s = divmod(s, 60)
    return f'{m}m{s:02d}s' if m else f'{s}s'

print('ready')

ready


## 1. EVAL_DIR 선택 (01 에서 생성)

In [2]:
fair_root = Path('experiments/fair_eval')
if fair_root.exists():
    for p in sorted(fair_root.iterdir()):
        if p.is_dir() and (p / 'config.json').exists():
            cfg = json.loads((p / 'config.json').read_text())
            ens_ok = (p / 'ensembles' / 'split_0' / 'c0' / 'meta.json').exists()
            smp_ok = (p / 'samples').exists()
            print(f'  {p.name}  K={cfg["K"]} methods={cfg["methods"]}  '
                  f'ens={"OK" if ens_ok else "--"}  samples={"OK" if smp_ok else "--"}')
else:
    print('  (없음 — 01 먼저 실행)')

  20260417_215937  K=10 methods=['Distance']  ens=OK  samples=OK
  20260418_221446  K=10 methods=['Distance']  ens=OK  samples=--
  20260419_081846  K=10 methods=['Distance']  ens=OK  samples=--
  20260419_171832_biodeg_n100_K10  K=10 methods=['Distance']  ens=OK  samples=--
  20260419_182917_biodeg_n100_K10_Dist-fix5  K=10 methods=['Distance']  ens=OK  samples=--
  20260420_063616_biodeg_n100_K10_Dist-fix5  K=10 methods=['Distance']  ens=OK  samples=--


In [3]:
_candidates = sorted([
    p for p in fair_root.iterdir()
    if p.is_dir() and (p / 'config.json').exists()
       and (p / 'ensembles' / 'split_0' / 'c0' / 'meta.json').exists()
])
EVAL_DIR = _candidates[-1] if _candidates else Path('experiments/fair_eval/NONE')
EVAL_DIR = Path('experiments/fair_eval/20260420_063616_biodeg_n100_K10_Dist-fix5')   # <-- legacy stock (재현 불가)
assert (EVAL_DIR / 'config.json').exists(), f'없음: {EVAL_DIR}'

config = json.loads((EVAL_DIR / 'config.json').read_text())
K = config['K']
classes = config['classes']
print(f'EVAL_DIR: {EVAL_DIR}')
print(f'  K={K}, methods={config["methods"]}, n_splits={config["n_splits"]}')
print(f'  dataset={config["dataset"]}, n_real={config["n_real"]}, classes={classes}')

# 01 의 Distance 설정에서 distance_negative_class 추출
method_params = config.get('method_params', {})
dist_cfg = method_params.get('Distance', {})
if dist_cfg.get('mode') == 'fixed':
    DIST_NEG = float(dist_cfg['value'])
    print(f'  distance_negative_class = {DIST_NEG} (01 config 에서 자동 추출)')
else:
    raise ValueError(
        f'01 이 Distance fixed 모드가 아님 (mode={dist_cfg.get("mode")}). '
        f'per-member 비교는 고정 distance 에서만 의미 있음.')

EVAL_DIR: experiments/fair_eval/20260420_063616_biodeg_n100_K10_Dist-fix5
  K=10, methods=['Distance'], n_splits=10
  dataset=biodeg, n_real=100, classes=[0, 1]
  distance_negative_class = 5.0 (01 config 에서 자동 추출)


## 2. TabEBM SGLD 하이퍼파라미터

K 멤버 모두 **같은 SGLD 파라미터**. seed 만 k 마다 다름 (01 의 corner_seed 와 일치시키기 위함).

In [4]:
# === TabEBM 기본 SGLD 하이퍼파라미터 (논문 Table 1 기본값) ===
SGLD_STEPS      = 200              # 논문 기본값
SGLD_STEP_SIZE  = 0.1              # 논문 기본값
SGLD_NOISE_STD  = 0.01             # 논문 기본값
STARTING_NOISE  = 0.01             # 논문 기본값

print(f'SGLD: steps={SGLD_STEPS}, step_size={SGLD_STEP_SIZE}, '
      f'noise={SGLD_NOISE_STD}, start_noise={STARTING_NOISE}')
print(f'distance_negative_class = {DIST_NEG} (01 과 동일)')
print(f'K = {K} members per split  →  seed = split_seed + k  (01 의 corner_seed 와 set-level 일치)')

SGLD: steps=200, step_size=0.1, noise=0.01, start_noise=0.01
distance_negative_class = 5.0 (01 과 동일)
K = 10 members per split  →  seed = split_seed + k  (01 의 corner_seed 와 set-level 일치)


## 3. 멤버 config 조립

K 개 멤버 각각 `tabebm_m{k}` 로 명명. 모두 같은 SGLD 설정 + 다른 seed_offset = k.

In [5]:
TABEBM_CONFIGS = [
    dict(
        name=f'tabebm_m{k}',
        sgld_steps=SGLD_STEPS,
        sgld_step_size=SGLD_STEP_SIZE,
        sgld_noise_std=SGLD_NOISE_STD,
        starting_point_noise_std=STARTING_NOISE,
        distance_negative_class=DIST_NEG,
        seed_offset=k,                  # worker: seed=split_seed + k (= 01 의 corner_seed)
    )
    for k in range(K)
]

print(f'{len(TABEBM_CONFIGS)} member configs:')
for c in TABEBM_CONFIGS[:3]:
    print(f'  {c["name"]:<15} seed_offset={c["seed_offset"]:<3d} dist={c["distance_negative_class"]:g}')
if len(TABEBM_CONFIGS) > 3:
    print(f'  ... (총 {len(TABEBM_CONFIGS)}개)')

10 member configs:
  tabebm_m0       seed_offset=0   dist=5
  tabebm_m1       seed_offset=1   dist=5
  tabebm_m2       seed_offset=2   dist=5
  ... (총 10개)


## 4. 병렬화 + 생성 설정

In [6]:

# === Split 선택 ===
# None = 전체 (canonical). list/int = subset (smoke test / 부분 재실행).
# SELECTED_SPLITS = None
# SELECTED_SPLITS = [0, 1]        # 2 split 만
SELECTED_SPLITS = 0             # 단일

GPUS            = [0, 1, 2, 3]
# GPUS          = [0, 1]
# GPUS          = [0]

PROCS_PER_GPU   = 7
# PROCS_PER_GPU = 2

# Storage: 500 per class (abundance). 04 evaluator slices: n_syn_total 500 → binary 250/class.
N_SYN_PER_CLASS = 500
# N_SYN_PER_CLASS = 250
# N_SYN_PER_CLASS = 10000          # N_SYN sweep 용

SKIP_EXISTING   = True

N_GPU = len(GPUS)
MAX_WORKERS = N_GPU * PROCS_PER_GPU
print(f'GPUs={GPUS}, workers={MAX_WORKERS}, N_syn_per_class={N_SYN_PER_CLASS}')
print(f'SKIP_EXISTING={SKIP_EXISTING}')


GPUs=[0, 1, 2, 3], workers=28, N_syn_per_class=500
SKIP_EXISTING=True


## 5. 샘플 생성 (병렬)

K × n_splits × classes 개 task. 각 task 는 `TabEBM.generate(X_tr[y==c], y==c, seed=split_seed+k)` 호출.

In [ ]:
from fair_eval_worker import run_one_sgld_task
from ensemble_ebm import apply_preprocessor, split_preprocessor_from_npz

data = np.load(EVAL_DIR / 'data.npz')
X_all_raw, y_all = data['X_all'], data['y_all']
sp = np.load(EVAL_DIR / 'splits.npz')

# SELECTED_SPLITS 정규화 (파일 상단 설정 사용)
if SELECTED_SPLITS is None:
    _gen_split_ids = list(range(config['n_splits']))
elif isinstance(SELECTED_SPLITS, int):
    _gen_split_ids = [SELECTED_SPLITS]
else:
    _gen_split_ids = list(SELECTED_SPLITS)
print(f'  generating for splits: {_gen_split_ids}  '
      f'({"canonical (all)" if len(_gen_split_ids)==config["n_splits"] else "DEBUG subset"})')
splits = [(i, sp[f'tr_{i}'], sp[f'val_{i}']) for i in _gen_split_ids]

# Paper B.3: per-split preprocessor 적용
X_all_per_split = {
    i: apply_preprocessor(X_all_raw, split_preprocessor_from_npz(sp, i))
    for i in _gen_split_ids
}

ens_dir = str(EVAL_DIR / 'ensembles')
samples_dir = EVAL_DIR / 'samples'
samples_dir.mkdir(exist_ok=True)

# tabebm_sample_config.json merge (기존 유지 + 신규 추가)
_tc_path = EVAL_DIR / 'tabebm_sample_config.json'
if _tc_path.exists():
    _existing = json.loads(_tc_path.read_text())
    _existing_names = {c['name'] for c in _existing.get('tabebm_configs', [])}
else:
    _existing = {'tabebm_configs': []}
    _existing_names = set()
_new_configs = [c for c in TABEBM_CONFIGS if c['name'] not in _existing_names]
_merged = _existing.get('tabebm_configs', []) + _new_configs
_tc_path.write_text(json.dumps({
    'tabebm_configs': _merged,
    'n_syn_per_class': N_SYN_PER_CLASS,
}, indent=2))
print(f'tabebm_sample_config.json: 기존 {len(_existing_names)}개 + 신규 {len(_new_configs)}개 = {len(_merged)}개')

# task 구성
tasks = []; skipped = 0; task_id = 0
for (split_i, tr, _val) in splits:
    X_all_s = X_all_per_split[split_i]
    for cfg in TABEBM_CONFIGS:
        for c in classes:
            out_path = samples_dir / f'split_{split_i}' / cfg['name'] / f'c{c}.npy'
            if SKIP_EXISTING and out_path.exists():
                skipped += 1; task_id += 1; continue
            gpu = GPUS[task_id % N_GPU]
            tasks.append(('single', split_i, c, cfg,
                          tr, X_all_s, y_all, N_SYN_PER_CLASS,
                          config['seed'], gpu, ens_dir))
            task_id += 1

n_total = len(tasks)
print(f'{n_total} tasks (skipped {skipped} existing), {MAX_WORKERS} workers')
if n_total == 0:
    print('모든 샘플 이미 존재 — 생성 건너뜀')
else:
    print()
    t0 = time.time(); done = 0
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(run_one_sgld_task, t): t for t in tasks}
        for f in as_completed(futs):
            r = f.result()
            out = samples_dir / f'split_{r["split_i"]}' / r['cfg_name']
            out.mkdir(parents=True, exist_ok=True)
            np.save(out / f'c{r["class_c"]}.npy', r['samples'])
            done += 1
            if done % 10 == 0 or done == n_total:
                elapsed = time.time() - t0
                eta = elapsed / done * (n_total - done)
                print(f'  [{done:>3d}/{n_total}]  '
                      f'elapsed {_fmt(elapsed)}  ETA {_fmt(eta)}', flush=True)
    print(f'\nDone -- {_fmt(time.time()-t0)}')


## 6. 검증

In [ ]:
expected = [c['name'] for c in TABEBM_CONFIGS]
ok = 0; fail = 0
for i in range(config['n_splits']):
    missing = []
    for cn in expected:
        for c in classes:
            if (samples_dir / f'split_{i}' / cn / f'c{c}.npy').exists():
                ok += 1
            else:
                missing.append(f'{cn}/c{c}'); fail += 1
    status = 'OK' if not missing else f'MISSING {missing[:3]}'
    print(f'  split_{i}: {status}')

print(f'\nTotal: {ok} OK, {fail} missing')
print(f'EVAL_DIR = {EVAL_DIR}  --> 04 에 입력')

## 7. (선택) Corner 일치 확인

내 TabEBM seed=split_seed+k 이 만든 corner 가 실제로 01 ensemble member k 의 corner 와 set-level 일치하는지 sanity check.

In [8]:
from tabebm.TabEBM import TabEBM

def to_set(arr):
    return frozenset(tuple(row) for row in arr)

SPLIT_CHECK = 0
split_seed = config['seed'] + SPLIT_CHECK * 100
all_match = True

print(f'split {SPLIT_CHECK} (split_seed={split_seed}) — 01 member k 의 corner vs TabEBM(seed=split_seed+k)')
print('=' * 70)
for k in range(K):
    # 01 member k class 0 corner 로드
    neg_path = EVAL_DIR / 'ensembles' / f'split_{SPLIT_CHECK}' / 'c0' / f'ebm_{k}' / 'negatives.npz'
    if not neg_path.exists():
        print(f'  [skip] 01 member {k} negatives.npz 없음')
        continue
    corner_01 = np.load(neg_path)['X_neg']

    # TabEBM.add_surrogate_negative_samples 가 같은 seed 에서 만드는 corner
    class_data = np.load(EVAL_DIR / 'ensembles' / f'split_{SPLIT_CHECK}' / 'c0' / 'class_data.npz')
    X_c = class_data['X_class']
    np.random.seed(split_seed + k)   # TabEBM.generate 내부의 seed_everything 과 같은 numpy 상태
    X_ebm_te, y_ebm_te = TabEBM.add_surrogate_negative_samples(
        X_c, distance_negative_class=DIST_NEG)
    corner_te = X_ebm_te[y_ebm_te == 1]

    match = to_set(corner_01) == to_set(corner_te)
    all_match &= match
    print(f'  member {k} (seed={split_seed+k}): 01 corner vs TabEBM corner set-equal = {match}')

if all_match:
    print('\n✅ 모든 확인된 member 에서 corner 일치 — 01 ensemble 과 완전 공정 비교 가능.')
else:
    print('\n⚠ 불일치 — ensemble_methods.py 의 RandomState 수정 확인 필요.')

split 0 (split_seed=42) — 01 member k 의 corner vs TabEBM(seed=split_seed+k)
  member 0 (seed=42): 01 corner vs TabEBM corner set-equal = True
  member 1 (seed=43): 01 corner vs TabEBM corner set-equal = True
  member 2 (seed=44): 01 corner vs TabEBM corner set-equal = True
  member 3 (seed=45): 01 corner vs TabEBM corner set-equal = True
  member 4 (seed=46): 01 corner vs TabEBM corner set-equal = True
  member 5 (seed=47): 01 corner vs TabEBM corner set-equal = True
  member 6 (seed=48): 01 corner vs TabEBM corner set-equal = True
  member 7 (seed=49): 01 corner vs TabEBM corner set-equal = True
  member 8 (seed=50): 01 corner vs TabEBM corner set-equal = True
  member 9 (seed=51): 01 corner vs TabEBM corner set-equal = True

✅ 모든 확인된 member 에서 corner 일치 — 01 ensemble 과 완전 공정 비교 가능.


## 8. Eval — K-union vs Single (Fair Budget)

**연구 질문**: 같은 synthetic 예산 (`N_SYN_TOTAL`) 을 하나의 TabEBM 에 다 몰아주는 것 vs K 멤버에 나눠주고 합집합으로 쓰는 것, 어느 쪽이 downstream classifier 에 더 도움될까?

**3가지 방식 비교 (6 classifier)**:
| 이름 | 설명 | 예산 |
|---|---|---|
| `baseline` | synthetic 없이 X_tr 만 | 0 |
| `single` | 02 의 `tabebm_single` 500 sample | N_total per class |
| `K_union` | `tabebm_m0..m{K-1}` 각각 N/K per class → 합집합 | **총합 동일** (fair budget) |
| `K_indiv` (mean±std) | `tabebm_m{k}` 각각 full budget 으로 classifier fit → K 개 bacc 의 평균/표준편차 | N_total × K (reference, not fair) |

**전제**:
- 02 를 먼저 돌려서 `tabebm_single` 이 `samples/` 에 있어야 `single` 열이 채워짐.
- 03_01 을 돌려서 `tabebm_m0..m{K-1}` 이 있어야 `K_union`/`K_indiv` 계산 가능.
- `SELECTED_SPLITS` 가 subset 이면 canonical 아님 — smoke test 용.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
from tabpfn import TabPFNClassifier
from ensemble_ebm import apply_preprocessor, split_preprocessor_from_npz
import pandas as pd

# === Eval 설정 ===
N_SYN_TOTAL_EVAL = 500   # paper 기본; per_class = N_SYN_TOTAL_EVAL // n_classes

# 6 classifier (04 와 동일 — paper author-code builders)
CLASSIFIER_BUILDERS_EVAL = {
    'knn':     lambda s: KNeighborsClassifier(n_jobs=-1),
    'lr':      lambda s: LogisticRegression(max_iter=1000, n_jobs=-1, random_state=s),
    'rf':      lambda s: RandomForestClassifier(n_jobs=-1, random_state=s),
    'xgboost': lambda s: xgb.XGBClassifier(n_jobs=-1, eval_metric='logloss',
                                            use_label_encoder=False, random_state=s),
    'mlp':     lambda s: MLPClassifier(max_iter=500, random_state=s),
    'tabpfn':  lambda s: TabPFNClassifier(n_estimators=1, device='auto', random_state=s,
                                           ignore_pretraining_limits=True),
}
CLASSIFIERS_EVAL = list(CLASSIFIER_BUILDERS_EVAL.keys())

# === 데이터 + split 로드 ===
data = np.load(EVAL_DIR / 'data.npz')
X_all_raw, y_all = data['X_all'], data['y_all']
sp = np.load(EVAL_DIR / 'splits.npz')
te_fixed = sp['te_fixed']
samples_root = EVAL_DIR / 'samples'

n_classes = len(classes)
n_total_pc  = max(1, N_SYN_TOTAL_EVAL // n_classes)        # single / K_indiv 용
n_member_pc = max(1, n_total_pc // K)                      # K_union 용 (fair budget)
print(f'N_SYN_TOTAL={N_SYN_TOTAL_EVAL}, n_classes={n_classes}, K={K}')
print(f'  single/K_indiv: {n_total_pc} per class')
print(f'  K_union:        {n_member_pc} per class per member × {K} = {n_member_pc*K} per class')

# === Split 리스트 (생성 단계에서 쓴 것과 동일) ===
eval_split_ids = _gen_split_ids   # 위 cell 에서 설정

rows = []
for split_i in eval_split_ids:
    prep  = split_preprocessor_from_npz(sp, split_i)
    X_all_s = apply_preprocessor(X_all_raw, prep)
    tr    = sp[f'tr_{split_i}']
    X_tr, y_tr = X_all_s[tr], y_all[tr]
    X_te, y_te = X_all_s[te_fixed], y_all[te_fixed]
    seed_i = config['seed'] + split_i

    split_dir = samples_root / f'split_{split_i}'
    single_dir = split_dir / 'tabebm_single'
    has_single = all((single_dir / f'c{c}.npy').exists() for c in classes)

    # K-union 구성 (fair budget: K × n_member_pc per class ≈ n_total_pc per class)
    union_per_class = {}
    for c in classes:
        parts = []
        for k in range(K):
            p = split_dir / f'tabebm_m{k}' / f'c{c}.npy'
            if not p.exists():
                raise FileNotFoundError(f'{p} missing — 03_01 gen 먼저')
            parts.append(np.load(p)[:n_member_pc])
        union_per_class[c] = np.vstack(parts)
    X_union = np.vstack([X_tr] + [union_per_class[c] for c in classes])
    y_union = np.concatenate([y_tr] + [np.full(len(union_per_class[c]), c) for c in classes])

    # Single 구성 (02 에서 생성된 tabebm_single 500 sample 에서 n_total_pc 슬라이스)
    if has_single:
        single_per_class = {c: np.load(single_dir / f'c{c}.npy')[:n_total_pc] for c in classes}
        X_single = np.vstack([X_tr] + [single_per_class[c] for c in classes])
        y_single = np.concatenate([y_tr] + [np.full(len(single_per_class[c]), c) for c in classes])

    # K-individual 구성 (참고용; 각 m{k} 독립 classifier fit, K 번 평균)
    indiv_per_k = []
    for k in range(K):
        mdir = split_dir / f'tabebm_m{k}'
        d_per_c = {c: np.load(mdir / f'c{c}.npy')[:n_total_pc] for c in classes}
        X_k = np.vstack([X_tr] + [d_per_c[c] for c in classes])
        y_k = np.concatenate([y_tr] + [np.full(len(d_per_c[c]), c) for c in classes])
        indiv_per_k.append((X_k, y_k))

    # === Classifier eval ===
    for clf_name in CLASSIFIERS_EVAL:
        row = {'split': split_i, 'classifier': clf_name}

        # baseline
        clf = CLASSIFIER_BUILDERS_EVAL[clf_name](seed_i)
        clf.fit(X_tr, y_tr)
        row['baseline'] = balanced_accuracy_score(y_te, clf.predict(X_te)) * 100

        # single
        if has_single:
            clf = CLASSIFIER_BUILDERS_EVAL[clf_name](seed_i)
            clf.fit(X_single, y_single)
            row['single'] = balanced_accuracy_score(y_te, clf.predict(X_te)) * 100
        else:
            row['single'] = np.nan

        # K-union (fair budget)
        clf = CLASSIFIER_BUILDERS_EVAL[clf_name](seed_i)
        clf.fit(X_union, y_union)
        row['K_union'] = balanced_accuracy_score(y_te, clf.predict(X_te)) * 100

        # K-individual (참고용)
        indiv_scores = []
        for X_k, y_k in indiv_per_k:
            clf = CLASSIFIER_BUILDERS_EVAL[clf_name](seed_i)
            clf.fit(X_k, y_k)
            indiv_scores.append(balanced_accuracy_score(y_te, clf.predict(X_te)) * 100)
        row['K_indiv_mean'] = float(np.mean(indiv_scores))
        row['K_indiv_std']  = float(np.std(indiv_scores))
        rows.append(row)

    print(f'  split {split_i} done')

# === 결과 DataFrame ===
eval_df = pd.DataFrame(rows)
print()
print('=== Per split x classifier ===')
print(eval_df.to_string(index=False))

# Per-classifier mean across splits (baseline / single / K_union / K_indiv)
print()
print('=== Mean across splits (classifier 별) ===')
mean_df = eval_df.groupby('classifier')[['baseline','single','K_union','K_indiv_mean']].mean().round(2)
print(mean_df)
print()
print('=== Delta vs baseline ===')
delta = mean_df.sub(mean_df['baseline'], axis=0).drop(columns='baseline').round(2)
delta.columns = [f'Δ_{c}' for c in delta.columns]
print(delta)

# Save
_out = EVAL_DIR / 'results' / f'03_01_eval_kunion_vs_single_n{N_SYN_TOTAL_EVAL}.csv'
_out.parent.mkdir(parents=True, exist_ok=True)
eval_df.to_csv(_out, index=False)
print(f'\nsaved: {_out}')
